# 22 — Pre-compute Satellite Grid for Deployment
## Lightweight lookup table replaces 1GB of raster files on Railway

**Problem:** Sentinel-2 rasters are ~1GB total — too heavy for Railway deployment.

**Solution:** Pre-compute NDVI/NDWI/NDBI at a 100m×100m grid across Berlin → save as small parquet file (~5MB). At prediction time, nearest grid point lookup replaces raster extraction.

**Output:** `data/processed/satellite_grid.parquet`

In [2]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from pyproj import Transformer
from pathlib import Path
from scipy.spatial import cKDTree
import hashlib

PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'

# Berlin bounding box (WGS84)
LAT_MIN, LAT_MAX = 52.34, 52.68
LON_MIN, LON_MAX = 13.08, 13.76
GRID_SPACING_M = 100  # 100m grid

# Convert to approximate grid dimensions
LAT_TO_M = 111_320
LON_TO_M = 111_320 * np.cos(np.radians(52.5))

n_lat = int((LAT_MAX - LAT_MIN) * LAT_TO_M / GRID_SPACING_M)
n_lon = int((LON_MAX - LON_MIN) * LON_TO_M / GRID_SPACING_M)

lats = np.linspace(LAT_MIN, LAT_MAX, n_lat)
lons = np.linspace(LON_MIN, LON_MAX, n_lon)

# Create grid of all points
grid_lats, grid_lons = np.meshgrid(lats, lons, indexing='ij')
grid_lats = grid_lats.flatten()
grid_lons = grid_lons.flatten()

print(f'Grid: {n_lat} × {n_lon} = {len(grid_lats):,} points')
print(f'Spacing: ~{GRID_SPACING_M}m')
print(f'Lat range: {LAT_MIN}–{LAT_MAX}, Lon range: {LON_MIN}–{LON_MAX}')

Grid: 378 × 460 = 173,880 points
Spacing: ~100m
Lat range: 52.34–52.68, Lon range: 13.08–13.76


In [3]:
# Load rasters and extract values at each grid point
RASTERS = {}
for idx_name in ['ndvi', 'ndwi', 'ndbi']:
    tif_path = PROC_DIR / f'{idx_name}_berlin.tif'
    if tif_path.exists():
        RASTERS[idx_name] = rasterio.open(tif_path)
        print(f'  {idx_name}: {RASTERS[idx_name].width}×{RASTERS[idx_name].height}, CRS={RASTERS[idx_name].crs}')

raster_crs = list(RASTERS.values())[0].crs
transformer = Transformer.from_crs('EPSG:4326', raster_crs, always_xy=False)

def extract_buffer_mean(raster, lat, lon, buffer_m):
    """Extract mean raster value within a circular buffer around a point."""
    x, y = transformer.transform(lat, lon)
    res_x = raster.res[0]
    buffer_pixels = int(buffer_m / res_x) + 1
    
    try:
        row, col = rowcol(raster.transform, x, y)
    except Exception:
        return np.nan
    
    if row < 0 or col < 0 or row >= raster.height or col >= raster.width:
        return np.nan
    
    r_min = max(0, row - buffer_pixels)
    r_max = min(raster.height, row + buffer_pixels + 1)
    c_min = max(0, col - buffer_pixels)
    c_max = min(raster.width, col + buffer_pixels + 1)
    
    window = rasterio.windows.Window.from_slices((r_min, r_max), (c_min, c_max))
    data = raster.read(1, window=window)
    
    rows_idx, cols_idx = np.mgrid[0:data.shape[0], 0:data.shape[1]]
    center_r = row - r_min
    center_c = col - c_min
    dist = np.sqrt(((rows_idx - center_r) * res_x) ** 2 + ((cols_idx - center_c) * abs(raster.res[1])) ** 2)
    mask = (dist <= buffer_m) & (data != 0) & np.isfinite(data)
    
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(data[mask]))

# Extract all 9 satellite features at each grid point
BUFFER_RADII = [100, 250, 500]
n_total = len(grid_lats)

print(f'\nExtracting satellite features for {n_total:,} grid points...')
print(f'({len(RASTERS)} indices × {len(BUFFER_RADII)} scales = {len(RASTERS) * len(BUFFER_RADII)} extractions per point)')

results = {
    'lat': grid_lats,
    'lon': grid_lons,
}

for idx_name, raster in RASTERS.items():
    for buffer_m in BUFFER_RADII:
        col_name = f'{idx_name}_{buffer_m}m'
        values = np.full(n_total, np.nan)
        
        for i in range(n_total):
            values[i] = extract_buffer_mean(raster, grid_lats[i], grid_lons[i], buffer_m)
            if (i + 1) % 50000 == 0:
                valid = np.isfinite(values[:i+1]).sum()
                print(f'  {col_name}: {i+1:,}/{n_total:,} ({100*valid/(i+1):.0f}% valid)')
        
        results[col_name] = values
        valid = np.isfinite(values).sum()
        print(f'  {col_name}: done ({valid:,}/{n_total:,} valid, {100*valid/n_total:.0f}%)')

grid_df = pd.DataFrame(results)
print(f'\nGrid DataFrame: {grid_df.shape}')

  ndvi: 10980×10980, CRS=EPSG:32633
  ndwi: 10980×10980, CRS=EPSG:32633
  ndbi: 10980×10980, CRS=EPSG:32633

Extracting satellite features for 173,880 grid points...
(3 indices × 3 scales = 9 extractions per point)
  ndvi_100m: 50,000/173,880 (87% valid)
  ndvi_100m: 100,000/173,880 (87% valid)
  ndvi_100m: 150,000/173,880 (87% valid)
  ndvi_100m: done (150,957/173,880 valid, 87%)
  ndvi_250m: 50,000/173,880 (87% valid)
  ndvi_250m: 100,000/173,880 (87% valid)
  ndvi_250m: 150,000/173,880 (87% valid)
  ndvi_250m: done (150,957/173,880 valid, 87%)
  ndvi_500m: 50,000/173,880 (87% valid)
  ndvi_500m: 100,000/173,880 (87% valid)
  ndvi_500m: 150,000/173,880 (87% valid)
  ndvi_500m: done (150,957/173,880 valid, 87%)
  ndwi_100m: 50,000/173,880 (87% valid)
  ndwi_100m: 100,000/173,880 (87% valid)
  ndwi_100m: 150,000/173,880 (87% valid)
  ndwi_100m: done (150,957/173,880 valid, 87%)
  ndwi_250m: 50,000/173,880 (87% valid)
  ndwi_250m: 100,000/173,880 (87% valid)
  ndwi_250m: 150,000/173,880

In [4]:
# Drop rows outside Berlin (all NaN satellite values = water/outside city boundary)
valid_mask = grid_df[['ndvi_100m', 'ndwi_100m', 'ndbi_100m']].notna().any(axis=1)
grid_clean = grid_df[valid_mask].copy()

# Fill remaining NaN with 0 (edges of coverage)
for col in grid_clean.columns:
    if col not in ['lat', 'lon']:
        grid_clean[col] = grid_clean[col].fillna(0)

print(f'Grid points total: {len(grid_df):,}')
print(f'Within Berlin (have satellite data): {len(grid_clean):,} ({100*len(grid_clean)/len(grid_df):.0f}%)')
print(f'Dropped (outside coverage): {len(grid_df) - len(grid_clean):,}')

# Save
grid_path = PROC_DIR / 'satellite_grid.parquet'
grid_clean.to_parquet(grid_path, index=False)
size_mb = grid_path.stat().st_size / 1e6
sha = hashlib.sha256(open(grid_path, 'rb').read()).hexdigest()[:16]

print(f'\nSaved: {grid_path.name}')
print(f'  Rows: {len(grid_clean):,}')
print(f'  Columns: {list(grid_clean.columns)}')
print(f'  File size: {size_mb:.1f} MB')
print(f'  SHA-256: {sha}')

# Verify: check a known location (Alexanderplatz)
alex_lat, alex_lon = 52.5219, 13.4132
grid_xy = np.column_stack([grid_clean['lon'].values * LON_TO_M, grid_clean['lat'].values * LAT_TO_M])
alex_xy = np.array([[alex_lon * LON_TO_M, alex_lat * LAT_TO_M]])
tree = cKDTree(grid_xy)
dist, idx = tree.query(alex_xy[0], k=1)

print(f'\nVerification — Alexanderplatz ({alex_lat}, {alex_lon}):')
print(f'  Nearest grid point: {dist:.0f}m away')
row = grid_clean.iloc[idx]
for col in ['ndvi_100m', 'ndvi_500m', 'ndwi_500m', 'ndbi_500m']:
    print(f'  {col}: {row[col]:.4f}')

Grid points total: 173,880
Within Berlin (have satellite data): 150,957 (87%)
Dropped (outside coverage): 22,923

Saved: satellite_grid.parquet
  Rows: 150,957
  Columns: ['lat', 'lon', 'ndvi_100m', 'ndvi_250m', 'ndvi_500m', 'ndwi_100m', 'ndwi_250m', 'ndwi_500m', 'ndbi_100m', 'ndbi_250m', 'ndbi_500m']
  File size: 10.5 MB
  SHA-256: f5e085ffa159d3ad

Verification — Alexanderplatz (52.5219, 13.4132):
  Nearest grid point: 32m away
  ndvi_100m: 0.0739
  ndvi_500m: 0.1293
  ndwi_500m: -0.1422
  ndbi_500m: 0.0123
